# CRAG Assistant — LangSmith Tracing & AWS Deployment

This notebook builds a **Corrective RAG (CRAG)** pipeline module by module, then wires in **LangSmith tracing** and ships it via **Docker to AWS (ECR → EC2)**. Scope-wise, this notebook covers two specific Phase 4 deliverables — observability and production deployment — not the full assignment (the golden dataset lives in a companion notebook).

## What's built here

- **CRAG pipeline**: retrieve → grade documents → generate, with a corrective fallback to live web search (Tavily) when retrieved context isn't good enough
- **Inline quality scoring**: every generation is scored for faithfulness, answer relevancy, and context precision via LLM-as-judge, immediately after generation — no separate eval pass needed
- **Full LangSmith tracing**: every request logs its question, retrieved chunks, generation, and quality scores, viewable per-thread in the LangSmith UI
- **Streamlit UI**, deployed via Docker to AWS EC2 through ECR — with an in-memory rate limiter and Streamlit's built-in `/_stcore/health` endpoint serving as the health check

## Why LangSmith instead of a custom dashboard

The assignment calls for a monitoring dashboard (volume, latency, top queries, RAGAS over time). Rather than building one from scratch, this project logs every trace to LangSmith and uses its project view directly — it already surfaces per-run latency, token cost, and full input/output/metadata without any additional code.

## Architecture


flowchart TD
    A([START]) --> B[Retrieve]

    B --> C[(FAISS Retriever)]
    C --> D[Grade Documents]

    D --> E{Relevant documents found?}

    E -->|Yes| F[Generate]
    E -->|No| G[Transform Query]

    G --> H[Web Search]
    H --> F

    F --> I([END])

## Stack

LangChain · LangGraph · LangSmith (tracing) · OpenAI (`gpt-4.1-mini` / `gpt-4o-mini`) · Tavily · FAISS · Streamlit · Docker · AWS (ECR, EC2)

## Requirements

A `.env` with `OPENAI_API_KEY`, `TAVILY_API_KEY`, `LANGSMITH_API_KEY`, `LANGCHAIN_TRACING_V2=true`, `LANGCHAIN_PROJECT`. See `requirements.txt` for dependencies.

In [1]:
import os

In [2]:
os.makedirs("src", exist_ok = True)

In [3]:
folder_path = "src/rag_metric"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/rag_metric


In [ ]:
%%writefile src/rag_metric/metrics.py

import os
from dotenv import load_dotenv
load_dotenv()  # loading all the environment variables

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "TESTING-phase"



from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable


# ============================================================
# Faithfulness: is every claim in the answer backed by context?
# ============================================================

class FaithfulnessGrade(TypedDict):
    # Explanation first -- forces the model to reason before it commits to a verdict.
    explanation: Annotated[str, ..., "Step-by-step reasoning, claim by claim"]
    faithful: Annotated[bool, ..., "True if every claim in the answer is supported "
        "by the retrieved context, False otherwise"]

faithfulness_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
faithfulness_grader = faithfulness_llm.with_structured_output(
    FaithfulnessGrade, method="json_schema", strict=True
)

faithfulness_instructions = """You are grading whether a STUDENT ANSWER is faithful
to the provided CONTEXT.

You will be given a CONTEXT (the retrieved documents) and a STUDENT ANSWER generated
from that context.

Grading criteria:
(1) Break the STUDENT ANSWER down into its individual factual claims.
(2) For each claim, check whether it can be directly inferred from the CONTEXT.
(3) A claim that is not supported by the CONTEXT, or that contradicts it, counts as
    unfaithful -- even if the claim happens to be true in general.
(4) The answer does not need to use every part of the CONTEXT. It only needs to be
    the case that whatever the answer DOES say is backed by the CONTEXT.

Faithfulness:
A faithfulness value of True means every claim in the STUDENT ANSWER is supported
by the CONTEXT.
A faithfulness value of False means at least one claim is unsupported or contradicts
the CONTEXT (a hallucination).

Explain your reasoning step by step, claim by claim, before giving your final answer.
Avoid stating your verdict at the outset."""

faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", faithfulness_instructions),
    ("human", "CONTEXT: {context}\n\nSTUDENT ANSWER: {answer}"),
])

@traceable(name="score_faithfulness")
def score_faithfulness(question: str, answer: str, chunks: list[dict]) -> bool:
    """chunks: list of {"content": ...} dicts"""
    context = " ".join(c["content"] for c in chunks)
    chain = faithfulness_prompt | faithfulness_grader
    grade = chain.invoke({"context": context, "answer": answer})
    return grade["faithful"]


# ============================================================
# Answer relevancy: does the answer address the question asked?
# ============================================================

class AnswerRelevancyGrade(TypedDict):
    explanation: Annotated[str, ..., "Step-by-step reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the answer directly and completely "
        "addresses the question, False otherwise"]

relevancy_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
relevancy_grader = relevancy_llm.with_structured_output(
    AnswerRelevancyGrade, method="json_schema", strict=True
)

answer_relevancy_instructions = """You are grading whether a STUDENT ANSWER is
relevant to the QUESTION that was asked.

Grading criteria:
(1) The answer should directly address what was asked, not a related but different
    question.
(2) The answer should not be incomplete -- dodging part of the question the context
    would have supported answering.
(3) The answer should not pad with information disconnected from the question.
(4) Do NOT grade factual correctness here. Only grade whether the response is on-topic
    and reasonably complete relative to the question.

Relevancy:
A relevant value of True means the answer is on-topic, addresses the question
directly, and is reasonably complete.
A relevant value of False means the answer is off-topic, evasive, or leaves out
something the question clearly asked for.

Explain your reasoning step by step before giving your final answer."""

answer_relevancy_prompt = ChatPromptTemplate.from_messages([
    ("system", answer_relevancy_instructions),
    ("human", "QUESTION: {question}\n\nSTUDENT ANSWER: {answer}"),
])

@traceable(name="score_answer_relevancy")
def score_answer_relevancy(question: str, answer: str) -> bool:
    chain = answer_relevancy_prompt | relevancy_grader
    grade = chain.invoke({"question": question, "answer": answer})
    return grade["relevant"]


# ============================================================
# Context precision: are the retrieved chunks actually needed?
# ============================================================

class ContextPrecisionGrade(TypedDict):
    explanation: Annotated[str, ..., "Reasoning for each chunk, in order"]
    chunk_relevance: Annotated[list[bool], ..., "One True/False per retrieved chunk, "
        "in the same order as given, indicating whether that chunk is relevant/necessary "
        "to answer the QUESTION"]

precision_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
precision_grader = precision_llm.with_structured_output(
    ContextPrecisionGrade, method="json_schema", strict=True
)

context_precision_instructions = """You are grading the PRECISION of retrieved
context for a RAG system.

You will be given a QUESTION and a numbered list of RETRIEVED CHUNKS returned by the
retriever.

Grading criteria:
(1) For each chunk, decide whether it is actually relevant and necessary for
    answering the QUESTION.
(2) A chunk on the same general topic but not needed to answer THIS question should
    be marked not relevant.
(3) Judge each chunk independently of the others and independently of its retrieval
    rank/position.
(4) If the QUESTION asks about something the source material does not cover, a chunk
    that explicitly states the requested information is absent or not specified
    counts as relevant -- it is the necessary evidence for a correct "not stated"
    answer, not noise.

Return one True/False verdict per chunk, in the same order as given.

Explain your reasoning for each chunk before giving your final verdicts."""

context_precision_prompt = ChatPromptTemplate.from_messages([
    ("system", context_precision_instructions),
    ("human", "QUESTION: {question}\n\nRETRIEVED CHUNKS:\n{chunks}"),
])

@traceable(name="score_context_precision")
def score_context_precision(question: str, chunks: list[dict]) -> float:
    chunks_string = "\n".join(f"[{i}] {c['content']}" for i, c in enumerate(chunks))
    chain = context_precision_prompt | precision_grader
    grade = chain.invoke({"question": question, "chunks": chunks_string})

    # LLM only judges each chunk's relevance; precision@k weighting is computed here,
    # matching RAGAS's actual formula
    flags = grade["chunk_relevance"]
    precisions_at_k, num_relevant = [], 0
    for k, is_relevant in enumerate(flags, start=1):
        if is_relevant:
            num_relevant += 1
            precisions_at_k.append(num_relevant / k)

    return sum(precisions_at_k) / len(precisions_at_k) if precisions_at_k else 0.0




Overwriting src/rag_metric/metrics.py


In [5]:
folder_path = "src/document_ingestion"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/document_ingestion


In [6]:
%%writefile src/document_ingestion/document_processor.py

from typing import List, Union
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    WebBaseLoader,
    TextLoader,
)


class DocumentProcessor:
    """Handles the document loading and processing"""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        """Initialise document processor"""
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""],
            length_function=len,
        )

    def load_from_url(self, url: str) -> List[Document]:
        """Load document from a URL or web address"""
        loader = WebBaseLoader(url)
        return loader.load()

    def load_from_pdf(self, file_path: Union[str, Path]) -> List[Document]:
        """Load PDF file"""
        loader = PyMuPDFLoader(str(file_path))
        return loader.load()

    def load_from_txt(self, file_path: Union[str, Path]) -> List[Document]:
        """Load document(s) from a TXT file"""
        loader = TextLoader(str(file_path), encoding="utf-8")
        return loader.load()

    def load_documents(self, sources: List[str]) -> List[Document]:
        """Load documents from a mix of URLs, .txt files, or PDF files and return them"""
        ## append = add one object
        ## extend = add many items
        docs: List[Document] = []
        for src in sources:
            if src.startswith("http://") or src.startswith("https://"):
                docs.extend(self.load_from_url(src))
            elif src.endswith(".pdf"):
                docs.extend(self.load_from_pdf(src))
            elif src.endswith(".txt"):
                docs.extend(self.load_from_txt(src))
            else:
                raise ValueError(
                    f"Unsupported source type: {src}. "
                    "Use a URL, .txt file, or .pdf file."
                )
        return docs

    def split_documents(self, documents: List[Document]) -> List[Document]:
        """
        Split documents into chunks
        Args:
            documents: List of documents to split
        Returns:
            List of split documents
        """
        return self.splitter.split_documents(documents)

    def process_documents(self, sources: List[str]) -> List[Document]:
        """
        Complete pipeline to load and split documents from any supported source type
        Args:
            sources: List of URLs, .txt file paths, or .pdf file paths to process
        Returns:
            List of processed document chunks
        """
        docs = self.load_documents(sources)
        return self.split_documents(docs)

Overwriting src/document_ingestion/document_processor.py


In [7]:
folder_path = "src/vectorstore"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/vectorstore


In [8]:
%%writefile src/vectorstore/vectorstore.py

from typing import List
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)


class VectorStore:
    """Creating VectorStore and its operations"""

    def __init__(self):
        """Initialise vectorstore with OpenAI embeddings"""
        self.embedding = OpenAIEmbeddings()
        self.vectorstore = None
        self.retriever = None

    def create_vectorstore(self, documents: List[Document], k: int = 4):
        """
        Receive documents and create the vector store
        k: Number of documents to retrieve
        """
        self.vectorstore = FAISS.from_documents(documents, self.embedding)
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})

    def get_retriever(self):
        """
        Get the retriever instance
        Returns:
            Retriever instance
        """
        if self.retriever is None:
            raise ValueError("Vector store not initialized. Call create_vectorstore first.")
        return self.retriever

    def retrieve(self, query: str) -> List[Document]:
        """
        Retrieve relevant documents for a query
        Args:
            query: Search query
        Returns:
            List of relevant documents
        """
        if self.retriever is None:
            raise ValueError("Vector store not initialized. Call create_vectorstore first.")
        return self.retriever.invoke(query)

Overwriting src/vectorstore/vectorstore.py


In [9]:
folder_path = "src/state"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/state


In [10]:
%%writefile src/state/rag_state.py


"""RAG state definition for LangGraph"""

from typing import List, Annotated
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph.message import add_messages


class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        web_search: whether to add search
        documents: list of retrieved documents
        messages: running chat history, auto-accumulated across turns via add_messages
    """
    question: str
    generation: str
    web_search: str
    documents: List[Document]
    messages: Annotated[list, add_messages]
    faithfulness: bool
    answer_relevancy: bool
    context_precision: float

Overwriting src/state/rag_state.py


In [11]:
folder_path = "src/node"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/node


In [12]:
%%writefile src/node/functions.py

"""Chains and tools shared across the CRAG graph's nodes."""

import os
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_tavily import TavilySearch

load_dotenv()  # pulls OPENAI_API_KEY / TAVILY_API_KEY from .env



# Helper: squash a list of Document objects into one context string

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



# Document relevance grader
class GradeDocuments(BaseModel):
    """Grade whether a retrieved document actually answers the user's question —
    not just whether it shares a topic or keywords with it."""

    binary_score: str = Field(
        description="Whether the document is relevant to the question: 'yes' or 'no'"
    )


grader_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,  # deterministic yes/no grading
)
structured_llm_grader = grader_llm.with_structured_output(GradeDocuments)

grader_system = """You are a grader assessing whether a retrieved document actually answers a user's \
question — not just whether it shares keywords or topic with it.

Grade 'yes' only if the document contains the specific information needed to answer the question. \
Grade 'no' if the document is only topically related (same general subject, overlapping terms or \
numbers) but doesn't contain the actual answer, or if it describes a different specific policy that \
happens to use similar language.

A document should never be marked 'yes' just because it mentions the same category of thing \
(e.g. a dollar amount, a time window, a department name) as the question — check that it's the same \
policy, not just the same shape of fact."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", grader_system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader



# Answer generation chain
generation_system_prompt = """
You are an assistant for question-answering tasks.

Use the following retrieved context and conversation history to answer the question.
If the answer is not contained in the context, say that you don't know.
Use no more than three sentences and keep the answer concise.

Chat history:
{chat_history}

Context:
{context}
"""

generation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generation_system_prompt),
        ("human", "{question}"),
    ]
)

# Separate LLM instance from the grader's — generation doesn't need structured output
generation_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

rag_chain = generation_prompt | generation_llm | StrOutputParser()



# Query rewriter (used before falling back to web search)

rewriter_system = """You a question re-writer that converts an input question to a better version that \
is optimized for web search. Look at the input and try to reason about the underlying semantic \
intent / meaning."""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rewriter_system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

rewriter_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
question_rewriter = re_write_prompt | rewriter_llm | StrOutputParser()



# Web search fallback tool
web_search_tool = TavilySearch(k=3)




Overwriting src/node/functions.py


In [13]:
%%writefile src/node/node.py



import os
from dotenv import load_dotenv


load_dotenv()  # pulls OPENAI_API_KEY / TAVILY_API_KEY from .env / LANGSMITH tracing


"""LangGraph nodes for the Corrective RAG workflow"""

from typing import Literal
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langsmith import traceable



from src.state.rag_state import GraphState
from src.node.functions import (
    format_docs,
    retrieval_grader,
    rag_chain,
    question_rewriter,
    web_search_tool,
)
from src.rag_metric.metrics import score_faithfulness, score_answer_relevancy, score_context_precision




class RAGNodes:
    """Contains node functions for the Corrective RAG workflow"""

    def __init__(self, retriever):
        """
        Initialize RAG nodes
        Args:
            retriever: Document retriever instance (built from any source type —
                       PDF, TXT, or URL — via DocumentProcessor + VectorStore)
        """
        self.retriever = retriever
        # Chains/tools are shared singletons defined once in functions.py —
        # imported directly here rather than re-passed through every constructor
        self.rag_chain = rag_chain
        self.retrieval_grader = retrieval_grader
        self.question_rewriter = question_rewriter
        self.web_search_tool = web_search_tool

    @traceable(name="retrieve_chunks", run_type="retriever")
    def retrieve_docs(self, state: GraphState) -> dict:
        """
        Retrieve relevant documents node
        Args:
            state: Current graph state
        Returns:
            Partial state update with retrieved documents
        """
        print("RETRIEVING FROM DATABASE")
        question = state["question"]
        documents = self.retriever.invoke(question)
        return {"documents": documents, "question": question}

    def grade_documents(self, state: GraphState) -> dict:
        """
        Grade each retrieved document for relevance.
        Web search is triggered only when none of the retrieved
        documents are relevant.
        Args:
            state: Current graph state with retrieved documents
        Returns:
            Partial state update with filtered documents and web_search flag
        """
        print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
        question = state["question"]
        documents = state["documents"]

        filtered_docs = []
        for document in documents:
            score = self.retrieval_grader.invoke(
                {"question": question, "document": document.page_content}
            )
            grade = score.binary_score.strip().lower()

            if grade == "yes":
                print("---GRADE: DOCUMENT RELEVANT---")
                filtered_docs.append(document)
            else:
                print("---GRADE: DOCUMENT NOT RELEVANT---")

        web_search = "Yes" if len(filtered_docs) == 0 else "No"

        if web_search == "Yes":
            print("---NO RELEVANT DOCUMENTS: WEB SEARCH REQUIRED---")
        else:
            print(f"---RELEVANT DOCUMENTS FOUND: {len(filtered_docs)}---")

        return {
            "documents": filtered_docs,
            "question": question,
            "web_search": web_search,
        }

    @traceable(name="generate_answer")
    def generate_answer(self, state: GraphState) -> dict:
        """
        Generate answer from retrieved documents node
        Args:
            state: Current graph state with (graded) retrieved documents
        Returns:
            Partial state update with the generated answer, RAGAS scores, and chat history
        """
        print("Generating the answer")
        question = state["question"]
        documents = state["documents"]
        messages = state.get("messages", [])

        generation = self.rag_chain.invoke(
            {
                "context": format_docs(documents),
                "question": question,
                "chat_history": messages,
            }
        )

        # Score against the same documents that were used to generate the answer.
        # All three are @traceable, so they nest under this node's own trace span
        # automatically -- no extra wiring needed.
        chunks_for_scoring = [{"content": d.page_content} for d in documents]
        faithful = score_faithfulness(question, generation, chunks_for_scoring)
        relevant = score_answer_relevancy(question, generation)
        precision = score_context_precision(question, chunks_for_scoring)

        return {
            "documents": documents,
            "question": question,
            "generation": generation,
            "faithfulness": faithful,
            "answer_relevancy": relevant,
            "context_precision": precision,
            "messages": [HumanMessage(content=question), AIMessage(content=generation)],
        }

    def transform_query(self, state: GraphState) -> dict:
        """
        Transform the query to produce a better question.
        Args:
            state: Current graph state
        Returns:
            Partial state update with the rewritten question
        """
        print("---TRANSFORM QUERY---")
        question = state["question"]
        documents = state["documents"]

        better_question = self.question_rewriter.invoke({"question": question})
        return {"documents": documents, "question": better_question}

    def web_search(self, state: GraphState) -> dict:
        """
        Web search based on the re-phrased question.
        Args:
            state: Current graph state
        Returns:
            Partial state update with web results appended to documents
        """
        print("---WEB SEARCH---")
        question = state["question"]
        documents = state["documents"]

        response = self.web_search_tool.invoke({"query": question})
        results = response["results"]
        top_results = results[:3]
        web_results = "\n\n".join(r["content"] for r in top_results)

        documents.append(Document(page_content=web_results))
        return {"documents": documents, "question": question}

    def decide_to_generate(self, state: GraphState) -> Literal["transform_query", "generate"]:
        """
        Route to query transformation when no relevant local
        documents remain. Otherwise, generate an answer.
        Args:
            state: Current graph state
        Returns:
            Name of the next node to route to
        """
        print("---ASSESS GRADED DOCUMENTS---")
        web_search = state["web_search"].strip().lower()

        if web_search == "yes":
            print("---DECISION: TRANSFORM QUERY---")
            return "transform_query"

        print("---DECISION: GENERATE---")
        return "generate"

Overwriting src/node/node.py


In [14]:
folder_path = "src/graph"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")



Created folder: src/graph


In [15]:
%%writefile src/graph/graph_builder.py

"""Graph builder for the Corrective RAG LangGraph workflow"""

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from src.state.rag_state import GraphState
from src.node.node import RAGNodes


class GraphBuilder:
    """Builds and manages the Corrective RAG LangGraph workflow"""

    def __init__(self, retriever):
        """
        Initialize graph builder
        Args:
            retriever: Document retriever instance — built upstream from any
                       mix of PDF, TXT, or URL sources via DocumentProcessor + VectorStore
        """
        self.nodes = RAGNodes(retriever)
        self.memory = MemorySaver()
        self.graph = None

    def build(self):
        """
        Build the Corrective RAG workflow graph
        Returns:
            Compiled graph instance
        """
        builder = StateGraph(GraphState)

        # Register nodes
        builder.add_node("retrieve", self.nodes.retrieve_docs)
        builder.add_node("grade_documents", self.nodes.grade_documents)
        builder.add_node("generate", self.nodes.generate_answer)
        builder.add_node("transform_query", self.nodes.transform_query)
        builder.add_node("web_search", self.nodes.web_search)

        # Fixed edges: START -> retrieve -> grade_documents -> (conditional)
        builder.add_edge(START, "retrieve")
        builder.add_edge("retrieve", "grade_documents")

        # Conditional edge: after grading, decide_to_generate picks the next node
        builder.add_conditional_edges(
            "grade_documents",
            self.nodes.decide_to_generate,
            {
                "transform_query": "transform_query",
                "generate": "generate",
            },
        )

        # Correction path: rewrite question -> web search -> generate -> END
        builder.add_edge("transform_query", "web_search")
        builder.add_edge("web_search", "generate")

        # Direct path
        builder.add_edge("generate", END)

        self.graph = builder.compile(checkpointer=self.memory)
        return self.graph

    def run(self, question: str, thread_id: str = "default") -> dict:
        """
        Run the Corrective RAG workflow
        Args:
            question: User question
            thread_id: Conversation thread ID — reuse the same ID across calls
                       so the checkpointer carries chat history forward
        Returns:
            Final state, including 'generation' (the answer) and 'messages'
        """
        if self.graph is None:
            self.build()

        config = {"configurable": {"thread_id": thread_id}}
        return self.graph.invoke({"question": question}, config=config)

Overwriting src/graph/graph_builder.py


In [16]:
folder_path = "src/pipeline"

# Create src/vectorstore, including parent folders
os.makedirs(folder_path, exist_ok=True)

# Create an empty __init__.py file
init_file = os.path.join(folder_path, "__init__.py")

with open(init_file, "a", encoding="utf-8"):
    pass
print(f"Created folder: {folder_path}")

Created folder: src/pipeline


In [17]:
%%writefile src/pipeline/ingestion_pipeline.py

"""End-to-end ingestion: load + chunk + embed any mix of PDF/TXT/URL sources
into a retriever, ready to hand to GraphBuilder."""

from typing import List, Union

from src.document_ingestion.document_processor import DocumentProcessor
from src.vectorstore.vectorstore import VectorStore


class IngestionPipeline:
    """Turns a list of sources (PDF paths, TXT paths, or URLs) into a retriever"""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50, k: int = 4):
        """
        Args:
            chunk_size: characters per chunk
            chunk_overlap: overlap between chunks
            k: number of documents the resulting retriever returns per query
        """
        self.processor = DocumentProcessor(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.vector_store = VectorStore()
        self.k = k

    def build_retriever(self, sources: Union[str, List[str]]):
        """
        Load, chunk, and embed the given sources; return a ready-to-use retriever.
        Args:
            sources: a single source or list of sources — any mix of .pdf, .txt
                     file paths, or http(s):// URLs
        Returns:
            A retriever instance (FAISS-backed)
        """
        chunks = self.processor.process_documents(sources)
        self.vector_store.create_vectorstore(chunks, k=self.k)
        return self.vector_store.get_retriever()

Overwriting src/pipeline/ingestion_pipeline.py


In [18]:
from src.pipeline.ingestion_pipeline import IngestionPipeline
from src.graph.graph_builder import GraphBuilder

pipeline = IngestionPipeline(chunk_size=500, chunk_overlap=50, k=5)
retriever = pipeline.build_retriever(["Aster_Vale_Labs_CRAG_Test_Handbook.pdf"])

graph_builder = GraphBuilder(retriever)
result = graph_builder.run("If a company laptop is stolen while an employee is traveling, who must be notified and within what timeframe?", thread_id="session-1")
print(result["generation"])

/Users/gabriel/Documents/data-sci-AI/phase-5/phase-5-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


RETRIEVING FROM DATABASE
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---RELEVANT DOCUMENTS FOUND: 1---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
Generating the answer
If a company laptop is stolen while an employee is traveling, IT Security must be notified within 30 minutes as per the company policy.


In [19]:
result

{'question': 'If a company laptop is stolen while an employee is traveling, who must be notified and within what timeframe?',
 'generation': 'If a company laptop is stolen while an employee is traveling, IT Security must be notified within 30 minutes as per the company policy.',
 'web_search': 'No',
 'documents': [Document(id='367d3ec9-bd54-40b5-9ceb-88d33a118c66', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-04T11:49:33+00:00', 'source': 'Aster_Vale_Labs_CRAG_Test_Handbook.pdf', 'file_path': 'Aster_Vale_Labs_CRAG_Test_Handbook.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'Aster Vale Labs CRAG Test Handbook', 'author': 'OpenAI', 'subject': 'Synthetic policy handbook for CRAG evaluation', 'keywords': '', 'moddate': '2026-08-04T11:49:33+00:00', 'trapped': '', 'modDate': "D:20260804114933+00'00'", 'creationDate': "D:20260804114933+00'00'", 'page': 4}, page_content='hours. A suspected theft of company equipment 

In [20]:
result['documents'][0].page_content

'hours. A suspected theft of company equipment must also be reported to IT Security within 30 minutes.\nExpense report timing\nExpense reports must be submitted within 15 calendar days after the trip ends. Reports submitted more than 60 days late\nrequire vice-president approval and may be denied.\nNote: The handbook does not specify which airline should be preferred for travel.'

In [21]:
%%writefile app.py



"""
Streamlit front-end for the Corrective RAG (CRAG) pipeline.

Concurrency model
------------------
Streamlit's server (Tornado) runs each connected browser session in its own
thread and gives it an isolated `st.session_state`. Per-user state (retriever,
graph, chat history) lives in `st.session_state` -- never in module-level
globals -- so sessions can't step on each other, and an unhandled exception
in one user's script run doesn't take the process down for anyone else.

The only intentionally shared, mutable state is the rate limiter's request
log below, which is guarded by a `threading.Lock`.

Health check
------------
Streamlit ships a built-in health endpoint at `/_stcore/health` -- no extra
route needed. Point a Docker HEALTHCHECK or k8s probe at
`http://<host>:8501/_stcore/health`.

Rate limiting
-------------
In-memory sliding-window limiter, keyed by session id, guarding the expensive
path (LLM calls). Process-local -- fine for a single container. If you scale
to multiple replicas later, swap `_request_log` for a Redis-backed counter.
"""

import os
import time
import uuid
import shutil
import threading
from collections import defaultdict, deque

import streamlit as st
from dotenv import load_dotenv

from src.pipeline.ingestion_pipeline import IngestionPipeline
from src.graph.graph_builder import GraphBuilder

load_dotenv()

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
RATE_LIMIT_MAX_REQUESTS = int(os.getenv("RATE_LIMIT_MAX_REQUESTS", "10"))
RATE_LIMIT_WINDOW_SECONDS = int(os.getenv("RATE_LIMIT_WINDOW_SECONDS", "60"))
UPLOAD_DIR = os.getenv("UPLOAD_DIR", "/tmp/crag_uploads")

st.set_page_config(page_title="CRAG Assistant", page_icon="📄", layout="wide")

# --------------------------------------------------------------------------
# Shared, thread-safe rate limiter
# --------------------------------------------------------------------------
_rate_lock = threading.Lock()
_request_log: dict[str, deque] = defaultdict(deque)


def check_rate_limit(session_id: str) -> tuple[bool, int]:
    """Sliding-window rate limit check. Returns (allowed, seconds_until_retry)."""
    now = time.monotonic()
    with _rate_lock:
        log = _request_log[session_id]
        while log and now - log[0] > RATE_LIMIT_WINDOW_SECONDS:
            log.popleft()
        if len(log) >= RATE_LIMIT_MAX_REQUESTS:
            retry_after = int(RATE_LIMIT_WINDOW_SECONDS - (now - log[0]))
            return False, max(retry_after, 1)
        log.append(now)
        return True, 0


# --------------------------------------------------------------------------
# Session state
# --------------------------------------------------------------------------
def init_session_state():
    defaults = {
        "session_id": str(uuid.uuid4()),
        "graph_builder": None,
        "retriever_ready": False,
        "chat_history": [],  # list[(role, content)]
        "last_metrics": None,
    }
    for key, value in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = value


init_session_state()

# --------------------------------------------------------------------------
# Sidebar: document ingestion
# --------------------------------------------------------------------------
with st.sidebar:
    st.header("📄 Knowledge source")

    uploaded_files = st.file_uploader(
        "Upload PDF or TXT files", type=["pdf", "txt"], accept_multiple_files=True
    )
    url_input = st.text_input("...or a URL", placeholder="https://...")

    chunk_size = st.number_input("Chunk size", value=500, min_value=100, max_value=4000, step=50)
    chunk_overlap = st.number_input("Chunk overlap", value=50, min_value=0, max_value=1000, step=10)
    top_k = st.number_input("Top-k retrieved chunks", value=5, min_value=1, max_value=20)

    if st.button("Build knowledge base", type="primary", use_container_width=True):
        sources = []
        session_upload_dir = os.path.join(UPLOAD_DIR, st.session_state.session_id)
        os.makedirs(session_upload_dir, exist_ok=True)

        for f in uploaded_files or []:
            path = os.path.join(session_upload_dir, f.name)
            with open(path, "wb") as out:
                out.write(f.getbuffer())
            sources.append(path)

        if url_input:
            sources.append(url_input.strip())

        if not sources:
            st.warning("Upload at least one file or provide a URL.")
        else:
            with st.spinner("Ingesting documents and building the retriever..."):
                try:
                    pipeline = IngestionPipeline(
                        chunk_size=chunk_size, chunk_overlap=chunk_overlap, k=top_k
                    )
                    retriever = pipeline.build_retriever(sources)

                    graph_builder = GraphBuilder(retriever)
                    graph_builder.build()

                    # Files are only needed on disk long enough for the loaders
                    # to read them -- their content is now inside the FAISS
                    # index in memory, so the temp copies can go.
                    shutil.rmtree(session_upload_dir, ignore_errors=True)

                    st.session_state.graph_builder = graph_builder
                    st.session_state.retriever_ready = True
                    st.session_state.chat_history = []
                    st.session_state.last_metrics = None
                    st.success(f"Knowledge base ready ({len(sources)} source(s)).")
                except Exception as e:
                    st.session_state.retriever_ready = False
                    st.error(f"Ingestion failed: {e}")

    st.divider()
    st.caption(f"Session: `{st.session_state.session_id[:8]}`")
    st.caption(f"Rate limit: {RATE_LIMIT_MAX_REQUESTS} requests / {RATE_LIMIT_WINDOW_SECONDS}s")

# --------------------------------------------------------------------------
# Main: chat
# --------------------------------------------------------------------------
st.title("Corrective RAG Assistant")

if not st.session_state.retriever_ready:
    st.info("Build a knowledge base from the sidebar to start chatting.")
else:
    for role, content in st.session_state.chat_history:
        with st.chat_message(role):
            st.markdown(content)

    question = st.chat_input("Ask a question about your documents...")

    if question:
        allowed, retry_after = check_rate_limit(st.session_state.session_id)

        if not allowed:
            st.chat_message("assistant").warning(
                f"Rate limit reached. Try again in {retry_after}s."
            )
        else:
            st.session_state.chat_history.append(("user", question))
            with st.chat_message("user"):
                st.markdown(question)

            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    try:
                        result = st.session_state.graph_builder.run(
                            question, thread_id=st.session_state.session_id
                        )
                        answer = result.get("generation", "I couldn't generate an answer.")
                        st.markdown(answer)

                        st.session_state.last_metrics = {
                            "faithfulness": result.get("faithfulness"),
                            "answer_relevancy": result.get("answer_relevancy"),
                            "context_precision": result.get("context_precision"),
                        }
                        st.session_state.chat_history.append(("assistant", answer))
                    except Exception as e:
                        # Confined to this session's thread -- won't affect
                        # other users or crash the server process.
                        error_msg = f"Something went wrong answering that: {e}"
                        st.error(error_msg)
                        st.session_state.chat_history.append(("assistant", error_msg))

    if st.session_state.last_metrics:
        with st.expander("Last answer's RAG metrics"):
            m = st.session_state.last_metrics
            c1, c2, c3 = st.columns(3)
            c1.metric("Faithfulness", "✅" if m["faithfulness"] else "❌")
            c2.metric("Answer relevancy", "✅" if m["answer_relevancy"] else "❌")
            precision = m["context_precision"]
            c3.metric("Context precision", f"{precision:.2f}" if precision is not None else "—")

      


Writing app.py


In [22]:
!streamlit run app.py

2026-08-12 20:57:21.138 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.178.59:8501

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            

✓ Installed:
  → /Users/gabriel/Documents/data-sci-AI/phase-5/phase-5-1/.agents/skills/developing-with-streamlit

✨ Successfully installed to /Users/gabriel/Documents/data-sci-AI/phase-5/phase-5-1

Note: Installed skills are symlinks to your local Streamlit environment.
      They generally should not be committed to git.

Recommended .gitignore snippet:
  # Streamlit agent skills (environment-specific symlinks)
  .agents/skills/developing-with-streamlit
USER_AGENT environment variable not set, consider setting it to identify your requests.
2026-08-

## Docker file

In [1]:
%%writefile  DockerFile

# ── Base image ─────────────────────────────────────────────────────────────
FROM python:3.11-slim

# ── Working directory ──────────────────────────────────────────────────────
WORKDIR /app

# ── Install dependencies ───────────────────────────────────────────────────
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy project files ─────────────────────────────────────────────────────
COPY . .

# ── Create folders in case they don't exist ────────────────────────────────
#RUN mkdir -p db data tools agent

# ── Expose Streamlit port ──────────────────────────────────────────────────
EXPOSE 8501

# ── Run the app ────────────────────────────────────────────────────────────
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]


Overwriting DockerFile


In [2]:
%%writefile Dockerfile

# ── Base image ─────────────────────────────────────────────────────────────
FROM python:3.11-slim

# Logs (print statements in node.py, etc.) flush immediately to docker logs
ENV PYTHONUNBUFFERED=1

# Install system dependencies (curl needed for HEALTHCHECK below)
RUN apt-get update && apt-get install -y \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

# ── Working directory ──────────────────────────────────────────────────────
WORKDIR /app

# ── Install dependencies ───────────────────────────────────────────────────
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy project files ─────────────────────────────────────────────────────
# .dockerignore keeps .env, .git, __pycache__ etc out of the image
COPY . .

# ── Run as non-root ────────────────────────────────────────────────────────
RUN useradd --create-home appuser
USER appuser

# ── Expose Streamlit port ──────────────────────────────────────────────────
EXPOSE 8501

# ── Health check (Streamlit's built-in endpoint) ───────────────────────────
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \
    CMD curl -f http://localhost:8501/_stcore/health || exit 1

# ── Run the app ────────────────────────────────────────────────────────────
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


In [3]:
%%writefile .dockerignore

.env
.pdf
.git
.gitignore
__pycache__
*.pyc
.venv
venv/
.streamlit
/tmp
*.ipynb
.ipynb_checkpoints

Overwriting .dockerignore


In [4]:
!docker buildx build --platform linux/amd64 -t crag-bot:latest .


[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 2.27kB                                     0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 2.27kB                                     0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 2.27kB                                     0.0s
 => [internal] load metadata for docker

In [1]:
# run — inject the key at runtime from your .env to acess as docker ignore keeps the .env file from getting to the image
!docker run -p 8501:8501 --env-file .env crag-bot



2026-08-12 21:19:56.726 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.17.0.2:8501
  External URL: http://92.208.187.79:8501

USER_AGENT environment variable not set, consider setting it to identify your requests.
RETRIEVING FROM DATABASE
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---RELEVANT DOCUMENTS FOUND: 1---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
Generating the answer
^C
  Stopping...
